# Task 5: Fine-Tuning BERT for POS Tagging & Chunking


### Author: Someshwar Waghmode

### Internship: Advanced Generative AI

### NOTE:
- This project follows a Hugging Face Transformer-based pipeline for sequence labeling using DistilBERT with proper token-label alignment evalution using seqeval metrics.

### Objective
TO Build and Fine-Tune a Transformer-based model (DistilBERT) for token classification tasks such as:
- Part-of-Speech (POS) Tagging
- Chunking (phrase Detection)

### Workflow
- Row Text
- Tokenization
- Label Alignment
- Model Training
- Evalution
- Inference
- Comparison

In [1]:
!pip install -q transformers datasets evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate

### Task 1: Dataset Selection
#### Dataset Used: wikiann(English)
### Reason:
- Publicly available multilingual dataset for token classification task
- Containe sequence labeling structure similar to POS tagging and chunking tasks
- Used here as a proxy dataset deu to computational efficiency in Colab
- Suitable for demonstrating a transformer-based token classification pipeline

### Label Categories
- O --> Outside entity
- B-PER / I-PER --> Person
- B-ORG / I-ORG --> Organization
- B-LOC / I-LOC --> Location

### IMP:
- wikinn is originally a named entity recognition (NER) dataset.Howevwe, in this assignment, it is usedto demonstrate a complete token classification pipeline inspired by SOP tagging and chunking task.

### Limitation:
- This dataset is originally designed for Named entity Recognition(NER), so it does not provode true SOP chunk lables. However, it is used here to demonstrate the token classification pipeline due to dataset accessibility and computational constraints.

In [3]:
# load Dataset
from datasets import load_dataset
dataset = load_dataset("wikiann", "en")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [4]:
# print dataset
print(dataset)

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [5]:
# print sample token for tarining
print("Sample Tokens: ", dataset["train"][0]["tokens"])
print("Sample Labels: ", dataset["train"][0]["ner_tags"])

Sample Tokens:  ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')']
Sample Labels:  [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0]


In [6]:
# Reduce dataset size for faster trainig
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_val = dataset["validation"].shuffle(seed=42).select(range(500))

dataset["train"] = small_train
dataset["validation"] = small_val


### Task 2: Data Preprocessing
#### Steps:
- Tokenization using DistilBERT tokenizer
- Aligning labels with tokenized output
- Handling subword tokens during tokenization
- Applying special token masking using -100

### Purpose:
- This is ensures that token-level labels are currectly aligned with subword tokenized inputs, enabling proper taring of the transformer model for sequencen labeling tasks

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# Tokenization function
def tokenize_and_align_labels(examples):
  tokenized_inputs = tokenizer(
      examples["tokens"],
      truncation=True,
      padding="max_length",
      is_split_into_words=True,
  )

  labels = []
  for i, label in enumerate(examples["ner_tags"]):
    word_ids = tokenized_inputs.word_ids(batch_index=i)
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
          label_ids.append(-100)
        elif word_idx != previous_word_idx:
          label_ids.append(label[word_idx])
        else:
          label_ids.append(-100)

        previous_word_idx = word_idx

    labels.append(label_ids)

  tokenized_inputs["labels"] = labels
  return tokenized_inputs



In [9]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [10]:
tokenized_dataset.set_format("torch")

In [11]:
print(dataset)

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 500
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 2000
    })
})


In [12]:
print(dataset["train"][0])

{'tokens': ["''", 'January', '21', "''", '–', 'Nanny', 'and', 'the', 'Professor'], 'ner_tags': [0, 0, 0, 0, 0, 1, 2, 2, 2], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['PER: Nanny and the Professor']}


In [13]:
# get labels name from datatset
label_list = dataset["train"].features["ner_tags"].feature.names

print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


### Task 3: Model Setup
#### Model Used:
- DistilBERT (lightweight transformer model)
- AutoModelForTokenClassification from Hugging Face Transformers

### Key Cnfig
- num_lables --> Number of unique output lables in the dataset
- label2id --> Mapping from label names to numeric IDs
- id2label --> Mapping from numeric IDs back to label names

### Purpose:
- These Config ensure correct alignment between model outputs and dataset labls for token classification tasks

In [14]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label = {i: l for i, l in enumerate(label_list)},
    label2id = {l: i for i, l in enumerate(label_list)}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
!pip install torch

In [16]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
print(device)

cpu


In [18]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [19]:
metric = evaluate.load("seqeval")

In [20]:
def compute_metrics(p):
  predictions, labels = p
  predictions = np.argmax(predictions, axis=2)

  true_label = [
      [label_list[l] for l in label if l != -100]
      for label in labels
  ]

  true_preds = [
      [label_list[p] for (p,l) in zip(pred, label) if l != --100]
      for pred, label in zip(predictions,labels)
  ]

  results = metric.compute(predictions=true_preds, references=true_labels)

  return {
      "precision": results["overall_precision"],
      "recall": results["overall_recall"],
      "f1": results["overall_f1"],
      "accuracy": results["overal_accuracy"]

  }

### Task 4: Trainig
#### Training Setup:
We use Hugging face Trainer API with the followind config
- Learning rate: 2e-5
- Number of epochs: 3
- Batch size: 16 (for training and evaluation )

#### Purpose:
- These settings are chosen to efficiently fine-tune the DistillBERT model while maintaining stable learninig and avoiding overfitting.

In [21]:
trainig_args = TrainingArguments(
    output_dir = "./results",
    learning_rate = 2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    logging_steps = 50,
)

In [22]:
trainer = Trainer(
    model = model,
    args = trainig_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.346622


In [ ]:
results = trainer.evaluate()
print(results)

### Task 5: Evaluation
#### evalution metric:
- we use the seqeval library for evaluating token classification perfoemance.
#### Metrics User
- Percision
- Recall
- F1 Score
#### Purpose:
- these metrics measure how well the model identifies and classifies tokens in sequence labeling tasks, providing a balanced evalution of performance.

#### Explanation of Metrics
- Percision: Measures how many predicted labels are correct out of all predicted labels
- Recall: Measures how many actual labels are currectly identified out of all true babels
- F1 Score: Harmonic mean of percision and recall , providing a balanced measure of perfoemance
- Accuracy: Overall percentage of correctly predicted labels

### Task 6: Inference
#### Objective:
- we test the tarined model on custom input sentences to evaluate its real-world prediction capability.

#### Process:
- Provide a custom input sentance
- Tokenize using the same DistilBERT the trained model
- pass the tokenized input through the trained model
- Decode predicted outputs into label names
- Display token-wise predicted labels

In [ ]:
sentence = "Someshwar work at Google in Pune"

# Tokenize properly
inputs = tokenizer(sentence, return_tensors="pt")

# Move input to same device as model
inputs = {k: v.to(device) for k, v in input.items()}

# get predictions
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=-1)

# Convert tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Map labels
predicted_labels = [label_list[p.item()] for p in predictions[0]]

# Print output ( ignore special tokens)
for token, label in zip(tokens, predicted_labels):
  if token not in ["[CLS]", "[SEL]"]:
    print(f"{token}: {label}")

## Observation:
- The model is tarined on the wikiannn dataset, which is orginally designed for Named Entity Recognition (NER) Therfore, the predictions primarily reflect named entity patterns ratger than stric POS Tagging or hunking lables

However, the complete pipeline correctly demonstrates the key components of token classification using tarnsformers models
like
- Tokenization
- Label agignment
- Transformer fine-tuning
- Sequence classification inference



### Task 7: Comparison ( POS Tahhing VS Chunking )
#### POS Tagging
- Word level tagging
- NN, VB, DT, JJ are in Output
- Lower Complex
- Used for grammar indentification
- Independent labeling
- EX: run - Verb

#### Chunking
- Phrase level grouping
- NP, VP, PP are in output
- Higher Complex
- Used for structure phase detection
- Depends on POS structure
- EX: New York - Noun Phrase (NP)

### Task 8: Report

#### Challenges Faced:
- Handling subword token alignment using word_ids()
- Managinig specil tokens using -100 masking
- Ensuring correct label mapping diring training and evalution

#### Technical Insights
- DistilBER provides a lightweight yet powerfull transformer architecture suirable for token classification tasks
- seqeval is essential for evaluting sequence labeling models effectively
- Proper preprocessing and label alignment significantly improve model performance

#### Final Observation:
- Transformer based model significantly outperform traditional approaches in sequence laneling tasks due to their ability to capture contextual relationships between word in a sentence

#### Future Improvement:
- This projectcan be extended by training on dedicated datasets such as Universal Dependencies ( for POS tagging ) or CoNLL - 2000 ( for chunking ) to achieve more task - specific and accurate results.